# Gate-Chain Optimizer Tutorial

This notebook walks through the functionality of the Gate-Chain Optimizer.

## Overview

The Gate-Chain Optimizer detects linear chains of 2-input gates (OR, AND, XOR, etc.) and replaces them with balanced trees. This reduces the critical path length from O(n) to O(log n).

**Before (Chain):**
```
in1 ──┐
      OR ──┐
in2 ──┘    OR ──┐
      in3 ──┘    OR ── out
            in4 ──┘
```
**Critical path**: 3 gates

**After (Balanced Tree):**
```
in1 ──┐
      OR ──┐
in2 ──┘    OR ── out
in3 ──┐    │
      OR ──┘
in4 ──┘
```
**Critical path**: 2 gates (log2(4) = 2)

## Two Approaches

The optimizer supports two methods for chain detection:

1. **Graph-based** - Builds a NetworkX graph and traverses it
2. **Netlist-based** - Works directly on the netlist without graph construction

| Aspect | Graph-based | Netlist-based |
|--------|-------------|---------------|
| Memory | Higher (full graph in RAM) | Lower (no graph) |
| Speed | Slower (graph construction) | Faster (direct access) |

## Class Architecture

The optimizer uses a clean class-based architecture:

| Class | Responsibility |
|-------|----------------|
| **GateConfig** | Configuration for a gate type (name, ntype, prefix) |
| **GateTreeBuilder** | Builds balanced gate trees from inputs |
| **ChainBoundaryExtractor** | Extracts inputs/outputs/internals of a chain |
| **ChainReplacer** | Orchestrates chain replacement |
| **GateChainScanner** | Subprocess-based chain detection |
| **CircuitOptimizer** | Main coordinator for the optimization pipeline |

## 1. Imports and Setup

In [ ]:
from netlist_carpentry import Module
from netlist_carpentry.core.enums.direction import Direction as Dir

from netlist_carpentry.routines.floodfill.chain_optimizer import (
    ChainBoundaryExtractor,
    ChainReplacer,
    CircuitOptimizer,
    GateChainScanner,
    GateTreeBuilder,
    find_gate_chains_netlist,
    get_gate_config,
    optimize_circuit,
    remove_instances,
    resolve_chain_instances,
)

from netlist_carpentry.utils.gate_lib import OrGate

## 2. Gate Configuration

The **GateConfig** dataclass defines which gate type to optimize.

In [ ]:
or_config = get_gate_config("or")

print(f"Name:         {or_config.name}")
print(f"Type Info:    {or_config.ntype_info}")
print(f"Chain Prefix: {or_config.chain_prefix}")
print(f"Gate Class:   {or_config.gate_cls}")

In [ ]:
and_config = get_gate_config("and")
print(f"AND Config: {and_config.name}, {and_config.ntype_info}")

xor_config = get_gate_config("§xor")
print(f"XOR Config: {xor_config.name}, {xor_config.ntype_info}")

## 3. Create Test Modules

Helper functions to create test modules with gate chains.
(This is for the notebook, not actually part of the optimizer code)

In [ ]:
def create_or_chain_module() -> Module:
    """
    Create a module with a 3-gate OR chain:
    in1, in2 -> OR1 -> OR2 (with in3) -> OR3 (with in4) -> out
    """
    m = Module(raw_path="or_chain_module")

    # Create input ports and wires
    p_in1 = m.create_port("in1_port", Dir.IN)
    p_in2 = m.create_port("in2_port", Dir.IN)
    p_in3 = m.create_port("in3_port", Dir.IN)
    p_in4 = m.create_port("in4_port", Dir.IN)

    w_in1 = m.create_wire("in1")
    w_in2 = m.create_wire("in2")
    w_in3 = m.create_wire("in3")
    w_in4 = m.create_wire("in4")

    # Connect input ports to wires
    m.connect(w_in1[0], p_in1[0])
    m.connect(w_in2[0], p_in2[0])
    m.connect(w_in3[0], p_in3[0])
    m.connect(w_in4[0], p_in4[0])

    # Create internal wires for chain
    w_or1_out = m.create_wire("or1_out")
    w_or2_out = m.create_wire("or2_out")

    # Create output
    w_out = m.create_wire("out")
    p_out = m.create_port("out_port", Dir.OUT)
    m.connect(w_out[0], p_out[0])

    # Create OR gates
    or1 = m.create_instance(OrGate, "or1")
    or2 = m.create_instance(OrGate, "or2")
    or3 = m.create_instance(OrGate, "or3")

    # Connect OR1: in1, in2 -> or1_out
    m.connect(w_in1[0], or1.ports["A"][0])
    m.connect(w_in2[0], or1.ports["B"][0])
    m.connect(w_or1_out[0], or1.ports["Y"][0])

    # Connect OR2: or1_out, in3 -> or2_out
    m.connect(w_or1_out[0], or2.ports["A"][0])
    m.connect(w_in3[0], or2.ports["B"][0])
    m.connect(w_or2_out[0], or2.ports["Y"][0])

    # Connect OR3: or2_out, in4 -> out
    m.connect(w_or2_out[0], or3.ports["A"][0])
    m.connect(w_in4[0], or3.ports["B"][0])
    m.connect(w_out[0], or3.ports["Y"][0])

    return m

## 4. Routines Floodfill Classes Deep Dive

### 4.1 ChainBoundaryExtractor

Extracts the boundary information (inputs, output, internal wires) from a chain.

In [ ]:
module = create_or_chain_module()
cfg = get_gate_config("or")

chains = find_gate_chains_netlist(module, cfg)
print(f"Found {len(chains)} chain(s)")
print(f"Chain keys: {chains[0]}")

In [ ]:
instances = resolve_chain_instances(module, chains[0])
print(f"Resolved {len(instances)} instances")

extractor = ChainBoundaryExtractor()
boundary, const_count, is_degenerate = extractor.extract_boundary(instances)

print("\nBoundary Info:")
print(f"  Inputs:         {len(boundary.inputs)}")
print(f"  Output:         {boundary.output.raw}")
print(f"  Internal wires: {len(boundary.internal_wires)}")
print(f"  Constants:      {const_count}")
print(f"  Is degenerate:  {is_degenerate}")

### 4.2 GateTreeBuilder

Builds a balanced tree from the extracted boundary.

In [ ]:
module = create_or_chain_module()
cfg = get_gate_config("or")

chains = find_gate_chains_netlist(module, cfg)
instances = resolve_chain_instances(module, chains[0])

extractor = ChainBoundaryExtractor()
boundary, _, _ = extractor.extract_boundary(instances)

remove_instances(module, instances)

builder = GateTreeBuilder(
    module=module,
    prefix="or_chain_1",
    boundary=boundary,
    cfg=cfg
)
builder.build()

print("Tree built successfully!")
print(f"New instances in module: {list(module.instances.keys())}")

### 4.3 ChainReplacer

The **ChainReplacer** class orchestrates the complete chain replacement workflow.

In [ ]:
module = create_or_chain_module()
cfg = get_gate_config("or")

chains = find_gate_chains_netlist(module, cfg)
print(f"Found chain: {chains[0]}")

replacer = ChainReplacer(boundary_extractor=ChainBoundaryExtractor())
info = replacer.replace_chain(
    module=module,
    chain_keys=chains[0],
    prefix="demo_chain",
    cfg=cfg
)

print("\nReplacement Result:")
print(f"  Status:      {info.status.name}")
print(f"  Num gates:   {info.num_gates}")
print(f"  Num inputs:  {info.num_inputs}")
print(f"  Was replaced: {info.was_replaced}")

### 4.4 GateChainScanner

The **GateChainScanner** uses subprocess-based scanning for memory safety on large designs.

In [ ]:
help(GateChainScanner)

### 4.5 CircuitOptimizer

The main coordinator class that ties everything together.

In [ ]:
help(CircuitOptimizer)

# 5. Main Entry Point: **optimize_circuit()**

The main function that combines everything. Uses subprocess-based scanning for memory safety on large designs. (the following is with netlist-based)

In [ ]:
help(optimize_circuit)

The following enables logging.

In [ ]:
from netlist_carpentry import CFG, initialize_logging

CFG.log_level = 2
initialize_logging()

### Example: Single Gate Type

In [ ]:
INPUT_FILE = "files/openMSP4301.v"
TOP_MODULE = "openMSP430"
OUTPUT_FILE = "single_gate_type_output.v"

result = optimize_circuit(
    input_path=INPUT_FILE,
    top_module=TOP_MODULE,
    gates="or",
    output_path=OUTPUT_FILE
)

### Example: Multiple Gate Types

In [ ]:
OUTPUT_FILE = "multiple_gate_type_output.v"

result = optimize_circuit(
    input_path=INPUT_FILE,
    top_module=TOP_MODULE,
    gates=["or", "and"],
    output_path=OUTPUT_FILE,
)

### Example: With Preprocessing
With **remove_degenerate=True**, gates with only constant inputs are removed before chain detection

In [ ]:
OUTPUT_FILE = "preprocessing_gate_type_output.v"

result = optimize_circuit(
    input_path=INPUT_FILE,
    top_module=TOP_MODULE,
    gates=["or"],
    output_path=OUTPUT_FILE,
    remove_degenerate=True,
)

### Evaluating Results
The **CircuitOptimizationResult** contains all statistics

In [ ]:
print(f"Modules processed:   {result.modules_processed}")
print(f"Modules with chains: {result.modules_with_chains}")
print(f"Chains detected:     {result.total_chains_detected}")
print(f"Chains replaced:     {result.total_chains_replaced}")
print(f"Chains skipped:      {result.total_chains_skipped}")
print(f"Chains failed:       {result.total_chains_failed}")

degenerate = result.all_degenerate()
problematic = result.all_problematic()
failed = result.all_failed()

---

## Summary: Class Hierarchy

```
optimize_circuit()                              # Main entry point
├── get_gate_config()                           # Gate configuration
└── CircuitOptimizer                            # Main coordinator
    ├── GateChainScanner                        # Subprocess-based scanning
    │   └── scan_module()                       # Scans single module
    │       └── find_gate_chains_netlist()      # Netlist-based detection
    │           ├── is_target_gate()
    │           └── build_gate_connectivity()
    │       # OR (alternative):
    │       └── find_gate_chains()              # Graph-based detection
    │           ├── is_gate()
    │           ├── is_chain_head()
    │           └── build_chain()
    └── ChainReplacer                           # Per chain
        ├── resolve_chain_instances()
        ├── ChainBoundaryExtractor
        │   ├── collect_external_inputs()
        │   ├── collect_output_wires()
        │   └── extract_boundary()
        ├── remove_instances()
        └── GateTreeBuilder
            └── build()
```

## Detection Method Comparison

| Function | Graph-based | Netlist-based |
|----------|-------------|---------------|
| Gate check | **is_gate(graph, node, cfg)** | **is_target_gate(instance, cfg)** |
| Find chains | **find_gate_chains(graph, cfg)** | **find_gate_chains_netlist(module, cfg)** |
| Connectivity | Built into graph | **build_gate_connectivity(module, cfg)** |
| Requires | **module.graph()** call | Direct instance access |